In [1]:
import torch
from sorl.gat_sim import GAT, GATConfig, BOS_TOKEN_ID
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup
# For fast training, set 'BOS_TOKEN_ID' to 15 in 'sorl/gat_sim.py' 

gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 6],  # 6 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    flex_kernel_options={
            "BLOCK_M": 32, "BLOCK_N": 32,
            "BLOCK_M1": 32, "BLOCK_N1": 64, "BLOCK_M2": 64, "BLOCK_N2": 32
        }
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

In [3]:
# ---- Copy & Paste Data Loader ----
from data.copy_paste import CopyPasteDataLoader

seq_len = 4
loader = CopyPasteDataLoader(vocab_size=16, max_token=10, seq_len=seq_len, device='cpu')
tokens, loss_mask = loader.get_batch(2)

In [ ]:
from sorl.neo_utils import sorl_search, compute_loss, sorl_evaluate

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
batch_size = 16
memory_span = 128
attn_blocksize = 1792
K = 2
assert seq_len % K == 0, f"seq_len {seq_len} must be divisible by K {K}"
max_iterations = 1
temperature = 2.0

for step in range(500): 
    optimizer.zero_grad()

    tokens, _ = loader.get_batch(batch_size)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv = sorl_search(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature)
    
    # --- compute loss ---
    traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize)
    loss = traj_loss + abs_loss
    # loss = traj_loss

    # GAPT
    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(tokens, model, n=4, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=10.0)
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}%")

validation step 0 | traj_loss: 2.36 | abs_loss: 3.04 | search adv: 0.72%
validation step 2 | traj_loss: 2.32 | abs_loss: 2.83 | search adv: 1.61%
validation step 4 | traj_loss: 2.27 | abs_loss: 2.46 | search adv: 2.08%
validation step 6 | traj_loss: 2.25 | abs_loss: 2.07 | search adv: 1.31%
validation step 8 | traj_loss: 2.23 | abs_loss: 1.73 | search adv: 1.42%
validation step 10 | traj_loss: 2.29 | abs_loss: 1.41 | search adv: 1.25%
validation step 12 | traj_loss: 2.33 | abs_loss: 1.13 | search adv: 2.53%
validation step 14 | traj_loss: 2.33 | abs_loss: 0.88 | search adv: 5.84%
validation step 16 | traj_loss: 2.40 | abs_loss: 0.68 | search adv: 5.59%
validation step 18 | traj_loss: 2.43 | abs_loss: 0.53 | search adv: 8.55%
validation step 20 | traj_loss: 2.46 | abs_loss: 0.42 | search adv: 8.97%
validation step 22 | traj_loss: 2.48 | abs_loss: 0.36 | search adv: 9.72%
validation step 24 | traj_loss: 2.52 | abs_loss: 0.33 | search adv: 10.08%
validation step 26 | traj_loss: 2.50 | abs

In [44]:
# generate function implementation 
# ----------------------------------
from sorl.gat_sim import generate

tokens, loss_mask = loader.get_batch(1)
idx = tokens[:, :1 + seq_len].clone()
print(f"init   | idx: {idx[0].tolist()}")
for i in range(seq_len+1): 
    idx = generate(model, idx, K=K, max_iterations=0, memory_span=memory_span, attn_blocksize=1792, temperature=0.0)
    print(f"step {i+1} idx: {idx[0].tolist()}")
copy_correct = torch.allclose(idx[0, 1 : 1 + seq_len], idx[0, 2 + seq_len : 2 + 2*seq_len])
print(f"Copy correct: {copy_correct}")

init   | idx: [15, 8, 3]
step 1 idx: [15, 8, 3, 17]
step 2 idx: [15, 8, 3, 17, 8]
step 3 idx: [15, 8, 3, 17, 8, 3]
Copy correct: True


In [ ]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0

        idx = tokens[pos : pos + sequence_length + 1].unsqueeze(0).to(device=device, dtype=torch.int32, non_blocking=True)
        pos += sequence_length
        yield idx

# ------------------------------------------------

# Question 1. Should we separate inputs / targets? 
#             That's really asking whether we want to 'reflect' on inputs, or inputs + next token
#             from the generation perspective, we ought to reflect on inputs and predict next-tok

# Reflection 1. 
# - based on above thought, we ought to modify the 'forward' method to take 'inputs' & 'targets' separately
#   the ._forward_pass and recursion should be done only on 'inputs'


data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))

train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=256, device="cpu")
val_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_val_000000.bin", sequence_length=256, device="cpu")

In [ ]:
# --- Benchmark Speed & Memory Cost --- 
from sorl.benchmark import run_benchmark_suite
import torch 


# Prepare data - TEST WITH SMALLER SEQUENCE FIRST
tokens = next(train_loader)

# Run benchmark
results = run_benchmark_suite(
    model, 
    tokens, 
    memory_span=1024, 
    num_runs=10
)